# 09 — Capstone: Fashion-MNIST, end to end

**Where we are.** You built a neural network from scratch, watched it break — vanishing gradients —
and repaired it with good initialization, dropout and normalization. Then you moved to PyTorch and
learned to turn every knob of a real model. This is the capstone: everything at once, on a real
image dataset, evaluated honestly.

**Prerequisites.** The whole chapter (NB 1–8) and the evaluation workflow you have run since the
beginning — a train/test split, baselines, a loss curve you *read*, a confusion matrix,
cross-validation, and the habit of reporting a number *with* its uncertainty (ch 00; the module-11
capstone).

**What you'll do.** Run a complete, honest workflow on **Fashion-MNIST**: look at the data → set
baselines → train a deep, He-initialized, dropout-regularized network in a scaled pipeline → read
its loss curve → study its confusion matrix and error gallery → measure seed-variance → put it in a
**fair fight** against tree ensembles → and reach a verdict.

The workflow itself is your reflex by now, so we move briskly through it. The genuinely new thing is
what sits *inside* the pipeline — a real deep network — and the verdict we reach. It may surprise
you. Everything is **shown to read**; the "Your turn" tasks at the end *modify* this pipeline.

## The dataset, and the honest question

**Fashion-MNIST** is 70 000 grayscale images, 28×28 pixels each, in 10 clothing categories —
T-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot. It was built as a
drop-in replacement for the classic MNIST handwritten digits: same size, same format, same number of
classes — but **harder**, because clothing varies more than digits do.

A capstone is not the place to ask "can a neural network fit this at all" — we already know it can.
The honest question, the one a practitioner actually faces, is sharper:

> **Is a deep network the right tool for this data — and how would we know?**

We answer it the only honest way: build the network well, evaluate it rigorously, and compare it —
fairly — against the alternatives.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from ml_course import datasets, viz
from ml_course.colors import CMAP_DIVERGING, COLORS

viz.use_course_style()

# Show the one-time dataset download (never silenced).
logging.basicConfig(level=logging.INFO, format="%(message)s")

SEED = 0


def set_determinism():
    # The reproducibility contract from NB 7: same seeds + deterministic kernels + a single
    # thread give bit-identical results across runs and machines.
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(1)


set_determinism()

CLASS_NAMES = datasets.FASHION_MNIST_CLASSES
X_train, X_test, y_train, y_test = datasets.fashion_mnist_subset(10000, 5000, seed=SEED)

print(f"train images: {X_train.shape}   test images: {X_test.shape}")
print(f"pixels per image: {X_train.shape[1]} (= 28 x 28), range [{X_train.min():.1f}, "
      f"{X_train.max():.1f}]")
print("train class balance:", np.bincount(y_train).tolist())

In [ ]:
# Fig 1 — one example from each class (reshape a 784-vector back to the 28 x 28 grid).
fig, axes = plt.subplots(2, 5, figsize=(10, 5.4))
for label, ax in enumerate(axes.ravel()):
    example = X_train[y_train == label][0].reshape(28, 28)
    ax.imshow(example, cmap="gray_r")
    ax.set_title(f"{label} — {CLASS_NAMES[label]}", fontsize=10)
    ax.axis("off")
fig.suptitle("Fashion-MNIST: one example per class")
fig.tight_layout(h_pad=2.5)
plt.show()

**Read the figure.** Ten clothing categories, each a 28×28 grayscale thumbnail. Two things
stand out. First, some classes are **visually distinctive** — a trouser, a bag, a sneaker have
shapes nothing else shares. Second, several classes are **genuinely ambiguous**: at this resolution a
*shirt*, a *T-shirt*, a *pullover* and a *coat* are four blurry torsos that even you might hesitate to
tell apart. Hold onto that — the model will confuse exactly the classes you do.

And notice what an image *is* here: a 28×28 **grid**. A pixel's meaning depends on its neighbours — an
edge, a sleeve, a hem is a *local* pattern in 2-D space. That spatial structure will turn out to be
the heart of our verdict.

## Baselines first — always anchor against the trivial

The oldest reflex in this course (ch 00): a number means nothing on its own. Before we celebrate any
accuracy, we need a floor to compare against. Two floors:

- **Majority-class** — always predict the most common label. With 10 balanced classes this scores
  about 0.10. Beating *this* only proves the model learned something rather than nothing.
- **A linear model** — logistic regression on the (scaled) pixels. This is the honest bar: whatever
  our deep network scores, it has to beat what a simple linear model already extracts.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
acc_dummy = accuracy_score(y_test, dummy.predict(X_test))

logistic = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
acc_logistic = accuracy_score(y_test, logistic.predict(X_test_s))

print(f"majority-class baseline:               {acc_dummy:.3f}")
print(f"logistic regression (scaled pixels):   {acc_logistic:.3f}")

**What this tells us.** The majority-class floor is 0.10, as expected for ten balanced
classes. But the linear model already reaches **0.80** — the pixels are far from random noise; a
straight decision boundary in 784-dimensional pixel space captures most of the signal.

So the bar is set, and it is high. "Deep learning earns its keep here" does **not** mean beating
chance — it means beating **0.80 by a margin that survives scrutiny.**

## The network — the thing under test

Now the model this whole chapter built toward: a deep, fully-connected network, assembled from pieces
you understand one at a time —

- **depth**: two hidden layers (784 → 256 → 128 → 10) that remap the pixels into a more separable
  representation (NB 2);
- **He initialization** so the signal survives the depth (NB 4);
- **dropout** to regularize (NB 5);
- **ReLU** activations (NB 3);
- trained with **Adam** (NB 8) in a **scaled pipeline** — a network needs standardized inputs (ch 11).

Every knob comes from a concept you built by hand. Here it is, **shown** — read it, then we run it.

In [ ]:
def build_net(in_dim=784, hidden=(256, 128), n_classes=10, p_drop=0.2, seed=SEED):
    # A fully-connected ReLU network: Linear -> ReLU -> Dropout blocks, then a linear read-out.
    # He-initialized weights (NB 4); dropout for regularization (NB 5).
    torch.manual_seed(seed)
    layers = []
    prev = in_dim
    for width in hidden:
        linear = nn.Linear(prev, width)
        nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")
        nn.init.zeros_(linear.bias)
        layers += [linear, nn.ReLU(), nn.Dropout(p_drop)]
        prev = width
    read_out = nn.Linear(prev, n_classes)
    nn.init.kaiming_normal_(read_out.weight, nonlinearity="linear")
    nn.init.zeros_(read_out.bias)
    layers.append(read_out)
    return nn.Sequential(*layers)


def train_net(X, y, epochs=30, batch_size=128, lr=1e-3, n_classes=10, seed=SEED):
    # The canonical loop from NB 7/8: shuffle, forward, cross-entropy, backward, Adam step.
    net = build_net(n_classes=n_classes, seed=seed)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    n = len(X_t)
    generator = torch.Generator().manual_seed(seed)
    loss_curve = []
    for _ in range(epochs):
        net.train()  # dropout ON
        order = torch.randperm(n, generator=generator)
        running = 0.0
        for start in range(0, n, batch_size):
            batch = order[start : start + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(net(X_t[batch]), y_t[batch])
            loss.backward()
            optimizer.step()
            running += loss.item() * len(batch)
        loss_curve.append(running / n)
    return net, loss_curve


def predict(net, X):
    net.eval()  # dropout OFF
    with torch.no_grad():
        return net(torch.tensor(X, dtype=torch.float32)).argmax(1).numpy()


set_determinism()
net, loss_curve = train_net(X_train_s, y_train)
y_pred = predict(net, X_test_s)
acc_net = accuracy_score(y_test, y_pred)

print(f"deep network test accuracy: {acc_net:.3f}")
print(f"training loss: {loss_curve[0]:.3f} (start) -> {loss_curve[-1]:.3f} (end, "
      f"{len(loss_curve)} epochs)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(loss_curve) + 1), loss_curve, color=COLORS["model"], marker="o",
        markersize=3)
ax.set_xlabel("epoch")
ax.set_ylabel("training loss (cross-entropy)")
ax.set_title("The network converges")
plt.show()

**Read the figure.** The training loss falls smoothly from ~0.80 to ~0.11 and flattens — a
**healthy, converged** descent. This is the shape you learned to want in NB 3 and NB 8: not the flat
line of a network stuck at chance (an optimization failure), but a steady drop that levels off
because the network has learned what this architecture can. (The curve plots each epoch's *average*
batch loss, so epoch 1 already sits below the untrained value of `ln 10 ≈ 2.3` — the network
descended during that first pass.)

And the payoff is real: **0.865** on held-out data, comfortably past the 0.80 linear baseline. Depth,
good initialization and Adam bought a genuine ~6-point gain. But a single accuracy number is a
summary — it hides *where* the model succeeds and fails. Let's look.

## Where does it err? The confusion matrix

Accuracy collapses 5 000 predictions into one number. The **confusion matrix** — a reflex since the
module-11 capstone — opens it back up: for every true class (row), where did the predictions
(columns) land? The diagonal is correct; everything off it is a mistake with a *direction*.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig = viz.plot_confusion_matrix(cm, CLASS_NAMES)
fig.axes[0].set_title("Confusion matrix — held-out 5 000 test images")
plt.setp(fig.axes[0].get_xticklabels(), rotation=45, ha="right")
plt.show()

# Per-class accuracy = correct / total for each true class.
per_class = cm.diagonal() / cm.sum(axis=1)
for label, name in enumerate(CLASS_NAMES):
    print(f"{label}  {name:12s} {per_class[label]:.3f}")

**Read the figure.** The diagonal is bright — most predictions are correct. The revealing
part is *where the off-diagonal mass sits*, and it is not random. It clusters among the **upper-body
garments**:

- **Shirt** is the hardest class by far — only **0.61** correct. It bleeds into T-shirt, coat and
  pullover.
- Pullover (0.77), T-shirt (0.83), coat (0.84) and dress (0.86) trade mistakes with one another.
- Meanwhile **trouser (0.98), bag (0.96), sneaker (0.95), sandal (0.93), ankle boot (0.91)** are
  almost never wrong.

The mistakes have a *shape*: the model confuses exactly the classes that look alike — the same ones
you hesitated over in Fig 1.

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f"misclassified: {len(wrong)} / {len(y_test)} ({100 * len(wrong) / len(y_test):.1f}%)")

rng = np.random.default_rng(SEED)
sample = rng.choice(wrong, size=15, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(10, 6.5))
for idx, ax in zip(sample, axes.ravel(), strict=False):
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"{CLASS_NAMES[y_test[idx]]}\n-> {CLASS_NAMES[y_pred[idx]]}", fontsize=9,
                 color=COLORS["error"])
    ax.axis("off")
fig.suptitle("Where the network goes wrong (true -> predicted)")
fig.tight_layout()
plt.show()

**Read the figure.** Look at the mistakes honestly. Most are images *you* would find hard: a
shirt called a T-shirt, a pullover called a coat, a coat called a dress. The captions read like
reasonable guesses, not blunders. The network is not broken — the **signal that separates these
classes is thin at 28×28 in grayscale.**

This is the first real clue to our verdict. The ceiling here may not be the model's *size* — it may
be the **representation**. We are asking a network to tell a shirt from a T-shirt after flattening
both into 784 unordered numbers.

## One split is one draw — how much does the seed matter?

The 0.865 above came from **one** random initialization. Change the seed and the weights start
somewhere else, the mini-batches shuffle differently, and the final accuracy moves a little. A single
number without its spread can mislead (the module-11 lesson) — so let's retrain and measure the
wobble.

In [ ]:
set_determinism()
seed_accs = []
for seed in range(10):
    net_s, _ = train_net(X_train_s, y_train, seed=seed)
    seed_accs.append(accuracy_score(y_test, predict(net_s, X_test_s)))
seed_accs = np.array(seed_accs)

print(f"10 seeds: mean {seed_accs.mean():.3f}, std {seed_accs.std():.3f}")
print(f"range [{seed_accs.min():.3f}, {seed_accs.max():.3f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.scatter(seed_accs, np.zeros_like(seed_accs), color=COLORS["model"], s=60, zorder=3,
           label="per-seed accuracy")
mean = seed_accs.mean()
ax.axvline(mean, color=COLORS["highlight"], lw=2, label=f"mean {mean:.3f}")
ax.axvspan(mean - seed_accs.std(), mean + seed_accs.std(), color=COLORS["highlight"],
           alpha=0.15, label="±1 std")
ax.set_yticks([])
ax.set_xlabel("test accuracy")
ax.set_title("Seed-variance: 10 trainings, same data")
ax.legend(loc="upper left", frameon=False)
plt.show()

**Read the figure.** Ten trainings on the *same* data land in a band about **1.4 points wide**
(roughly 0.858–0.871), purely from the random seed. The mean sits near 0.863; our single-split 0.865
was a typical draw.

This band is our **ruler**. When we compare methods next, any gap **narrower than ~0.01 is inside the
noise** — a tie, not a win. Keep that in mind for what follows.

## Is a deep network even the right tool? A fair fight

A capstone earns its verdict by comparing honestly. So we put the network against two strong
non-neural methods you already know — **Random Forest** and **HistGradient Boosting** — under
identical conditions:

- **same data**, the same 5-fold cross-validation, the same accuracy metric;
- but the **trees see the raw pixels** and the **network gets its scaled pipeline** — because trees
  need no scaling and a network does. That preprocessing difference is not cheating; it *is* the
  honest comparison — each method with the preprocessing it actually needs.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
splits = list(folds.split(X_train, y_train))

# The network: scale inside each fold (fit on the fold's train part only), then train.
set_determinism()
mlp_scores = []
for tr, va in splits:
    fold_scaler = StandardScaler().fit(X_train[tr])
    net_cv, _ = train_net(fold_scaler.transform(X_train[tr]), y_train[tr])
    preds = predict(net_cv, fold_scaler.transform(X_train[va]))
    mlp_scores.append(accuracy_score(y_train[va], preds))
mlp_scores = np.array(mlp_scores)

# The trees: raw pixels, no scaling.
rf_scores, hg_scores = [], []
for tr, va in splits:
    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rf.fit(X_train[tr], y_train[tr])
    rf_scores.append(accuracy_score(y_train[va], rf.predict(X_train[va])))
    hg = HistGradientBoostingClassifier(random_state=SEED)
    hg.fit(X_train[tr], y_train[tr])
    hg_scores.append(accuracy_score(y_train[va], hg.predict(X_train[va])))
rf_scores, hg_scores = np.array(rf_scores), np.array(hg_scores)

for name, s in [("MLP (scaled)", mlp_scores), ("Random Forest (raw)", rf_scores),
                ("HistGB (raw)", hg_scores)]:
    print(f"{name:22s} {s.mean():.3f} +/- {s.std():.3f}")

In [ ]:
methods = ["MLP\n(scaled)", "Random Forest\n(raw)", "HistGB\n(raw)"]
means = [mlp_scores.mean(), rf_scores.mean(), hg_scores.mean()]
stds = [mlp_scores.std(), rf_scores.std(), hg_scores.std()]
# Highlight the winner; the others in the neutral model colour.
best = max(means)
bar_colors = [COLORS["highlight"] if m == best else COLORS["model"] for m in means]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(methods, means, yerr=stds, capsize=6, color=bar_colors)
ax.set_ylabel("5-fold CV accuracy")
ax.set_ylim(0.80, 0.90)
ax.set_title("A fair fight: deep network vs. tree ensembles")
for i, m in enumerate(means):
    ax.text(i, m + stds[i] + 0.002, f"{m:.3f}", ha="center")
plt.show()

**Read the figure — and the verdict.** Here is the honest result:

- **HistGradient Boosting: 0.876** — the winner.
- **Deep network (MLP): 0.863** and **Random Forest: 0.855** — effectively tied (the network edges RF
  in every fold, but by less than the ~0.01 noise band).

So the deep network is **competitive, but not the best.** HistGB wins by about 1.3 points — a real
gap: it is roughly three times the seed noise and holds in all five folds — and it does so on the
**raw** pixels, with no scaling to worry about.

And accuracy is not the only axis. The trees also skipped the standardization step the network
needed, and a full accounting would weigh **training cost** too — timing each `fit` under matched
settings, a worthwhile exercise in its own right. The honest summary is not "one model is best" but
"each has a different profile of accuracy, preprocessing and cost."

This is the through-line of the whole course, stated one last time on real data: **there is no
universal best model.**

## Two honest lessons — and what the data is really asking for

There are **two** separate things to read from that bar chart, and it is worth keeping them apart.

**First: why does a boosted tree edge out our network?** Not because of images, as it turns out —
both models see the same flattened 784 pixels, and a tree split on "pixel 387 > 0.4" is exactly as
blind to which pixels are neighbours as the network is. This is the recurring lesson from the
ensembles chapters: on flat, tabular problems, gradient-boosted trees are formidable, and here they
edge out this particular network. (On the smaller 8×8 digits of module 11, the same contest was a
*tie* — so "a tree beats a dense net on pixels" is a dataset-specific result, not a law.)

**Second — and the finale's real punchline — what are *all three* models leaving on the table?**
Every one of them, tree and network alike, flattened the 28×28 grid into a row and **threw away which
pixels are next to each other**. To all of them, "the pixel at (3, 4)" and "the pixel at (3, 5)" are
unrelated inputs. The distinction between a shirt and a T-shirt lives in *spatial* structure — local
edges, sleeves, hems — and none of these models is built to use it.

That is what the next architecture fixes. A **convolutional network** (CNN) slides small filters
across the grid, so *locality* and *weight-sharing* come for free instead of being learned from
scratch — the right **inductive bias** for images, and exactly where the next stage of your learning
points. We can even look at what our dense network's first layer learned — and, tellingly, at what it
did *not*.

In [ ]:
# Each of the 256 first-layer neurons has a 784-weight vector -> reshape to a 28 x 28 "filter".
W1 = net[0].weight.detach().numpy()
limit = np.abs(W1).max()

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for row, ax in enumerate(axes.ravel()):
    ax.imshow(W1[row].reshape(28, 28), cmap=CMAP_DIVERGING, vmin=-limit, vmax=limit)
    ax.axis("off")
fig.suptitle("What the first layer learned — and did not (12 of its 256 filters)")
fig.tight_layout()
plt.show()

**Read the figure.** Each tile is one first-layer neuron's 784 weights, folded back into the
28×28 image shape (coral = negative, teal = positive). And here is the honest, telling result: they
look **mostly like noise**. You do *not* see the crisp edge- and stroke-detectors that a
convolutional network's first layer famously learns — only faint, scattered large-scale structure.

That absence *is* the point. A dense network has **no built-in reason** to prefer local, spatial
features: it treats the 784 pixels as an unordered list, so its weights spread across the whole
image with nothing to organize them into edges. A convolutional network **builds** locality and
weight-sharing in — which is why its filters *do* become clean edge and texture detectors, and why
it is the right tool for images. That is the bridge to where this all goes next.

In [ ]:
# "No universal best" the other way: on clean tabular data, the network is fully competitive.
# breast_cancer (module 11): 30 homogeneous numeric features, 5-fold CV.
from sklearn.model_selection import cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

bc = datasets.load_breast_cancer()
X_bc, y_bc = bc.drop(columns="target"), bc["target"]
bc_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
bc_models = {
    "MLP (scaled)": make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=SEED),
    ),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=SEED),
    "HistGB": HistGradientBoostingClassifier(random_state=SEED),
}
for name, model in bc_models.items():
    scores = cross_val_score(model, X_bc, y_bc, cv=bc_cv)
    print(f"{name:16s} {scores.mean():.3f} +/- {scores.std():.3f}")

## "No universal best" — the other direction

One guard against over-learning the lesson. It would be easy to walk away thinking "trees beat neural
networks." That is the **wrong** takeaway. The right one is structural: **match the tool to the shape
of the data.**

The cell above ran the same three methods on a *different* dataset — the Wisconsin breast-cancer
measurements from module 11: 30 clean, homogeneous numeric features. The standing **reverses**: the
MLP (0.979) is now at least the equal of the trees (0.970 / 0.965), where on the pixels it lost to
HistGB by a real margin. Holding ourselves to our own ruler, that top gap is *within* the ~0.01
noise band — so read it as "the dense net is now fully competitive," not a decisive win. But that
itself is the point: the model that *lost* on images is right back in the lead group here.

Same three methods, a data-dependent order. Neither is universally better; each fits a different kind
of data. Dense networks are at home on homogeneous numeric vectors, gradient-boosted trees on tabular
problems broadly, and — for images — the spatial structure calls for a CNN.

## What you built

You ran a **complete, honest, end-to-end machine-learning workflow** — the kind a practitioner
actually does — on a real image dataset:

- looked at the data and formed an expectation;
- set **baselines** (chance and a linear model) so every later number had meaning;
- trained a **deep, He-initialized, dropout-regularized network** in a scaled pipeline, and read its
  **loss curve** to confirm it converged;
- opened up the accuracy with a **confusion matrix** and an **error gallery**, and found the mistakes
  honest and structured;
- measured **seed-variance** so you knew the size of the noise;
- ran a **fair cross-method comparison** and reached a verdict you can defend: a boosted tree edges
  out the network here, and — the deeper point — *every* flat-pixel model leaves the image's spatial
  structure unused, which is exactly what a CNN is built to exploit.

You are no longer running a model — you are *evaluating* one, and reasoning about *why* it behaves as
it does. That is the whole game.

## Where this goes next

**NB 10** closes the course: where machine learning goes from here — convolutional networks for
images, and the sequence models (RNNs, transformers) behind today's language systems — and a look
back across the twelve methods you have built, from k-nearest neighbours to neural networks.

## Your turn

Each task **modifies the pipeline above** — change one thing, rerun, and read what happens.

- **(warm-up)** Raise the dropout to `nn.Dropout(0.5)` in `build_net` and retrain the main network.
  Does stronger regularization move the ~0.865, or is the ceiling somewhere the dropout rate cannot
  reach?
- **(core)** Give the network more capacity — add a third hidden layer (say `hidden=(256, 128, 64)`)
  **or** train for 60 epochs instead of 30 — and watch the training loss *and* the test accuracy.
  Does more capacity or more training beat 0.865, or does the train–test gap widen (overfitting)?
- **(reach)** Retrain on **only the four upper-body classes** — labels **0 (T-shirt), 2 (Pullover),
  4 (Coat), 6 (Shirt)**. Filter the train and test sets to these; two edits make it run: **relabel**
  them to 0–3 (so cross-entropy sees contiguous classes) and train with **`train_net(..., n_classes=4)`**.
  Rebuild the confusion matrix on the four. If accuracy stays low even head-to-head, the confusion is
  **intrinsic** to 28×28 pixels — not a shortage of classes. Connect what you find back to the
  argument for a CNN.

## References

- Xiao, H., Rasul, K., & Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset for Benchmarking
  Machine Learning Algorithms.* arXiv:1708.07747.
- LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to document
  recognition. *Proceedings of the IEEE*, 86(11), 2278–2324. https://doi.org/10.1109/5.726791
- He, K., Zhang, X., Ren, S., & Sun, J. (2015). Delving deep into rectifiers: surpassing human-level
  performance on ImageNet classification. *ICCV*. arXiv:1502.01852. *(initialization, NB 4)*
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: a
  simple way to prevent neural networks from overfitting. *JMLR*, 15, 1929–1958. *(NB 5)*
- Kingma, D. P., & Ba, J. (2015). Adam: a method for stochastic optimization. *ICLR*. arXiv:1412.6980.
  *(NB 8)*
- Paszke, A., et al. (2019). PyTorch: an imperative style, high-performance deep learning library.
  *NeurIPS*.

*Previous: 12.8 — the model and its parameters. Next: 12.10 — where ML goes next, and the whole
course.*